# AlphaGenome CRE Experiments

A unified notebook for running **systematic CRE (cis-regulatory element) perturbation experiments** with the [AlphaGenome API](https://github.com/google-deepmind/alphagenome).

**Quick start:** Fill in the gene name and cell type in section 1, adjust settings in section 2 if needed, then hit **Runtime → Run all**.

## What are CREs?

Cis-regulatory elements — enhancers, promoters, silencers — are stretches of non-coding DNA that act as molecular switches, controlling *when* and *where* genes are turned on or off. Understanding which CREs regulate a given gene is fundamental to interpreting non-coding genetic variation and disease mechanisms.

## What is *in silico* mutagenesis (ISM)?

Instead of performing wet-lab experiments, we computationally edit DNA sequences and use a trained model (AlphaGenome) to predict the effect on gene expression. Think of it as a virtual experiment: we can systematically perturb every region around a gene and measure which perturbations matter — all without touching a pipette.

## Why would I use this?

- **Understand which enhancers regulate your favourite gene** in a specific cell type or cancer cell line.
- **Interpret GWAS hits in non-coding regions** — identify which regulatory elements a variant disrupts.
- **Map the regulatory landscape** around a disease-associated gene to prioritize functional follow-up.

## What this notebook does

We implement three [CREME](https://www.nature.com/articles/s41588-024-01923-3)-style experiments (Toneyan and Koo, Nature Genetics 2024):

1. **Necessity test** — Tile a locus, shuffle each tile, identify essential CREs.
2. **Higher-order interaction** — Greedy ablation to discover CRE cooperativity.
3. **CRISPRi tiling scan** — Fine-resolution shuffle scan with full track outputs.

### How deletion is simulated

We cannot truly "delete" DNA *in silico* — removing bases would change the sequence length and shift the reading frame of the model. Instead, we simulate deletion using **dinucleotide shuffling**: the reference sequence of each tile is replaced with a randomly permuted version that preserves the exact frequencies of all 16 dinucleotides (AA, AC, AG, AT, ...). This destroys transcription factor binding sites and regulatory grammar while keeping GC content and local sequence composition intact, providing a biologically meaningful null. Each tile is shuffled multiple times and scored in both forward and reverse-complement orientations; the effect is averaged across all replicates to produce a robust estimate. This is the same perturbation strategy used by [CREME](https://www.nature.com/articles/s41588-024-01923-3).

## Setup

### AlphaGenome API key (required)

One-time setup:
1. **Get an API key**: visit the [AlphaGenome API page](https://aistudio.google.com/apikey) and create a new key.
2. **Add it to Colab Secrets**: click the **key icon** in the left sidebar, then **+ Add new secret**. Set the name to `ALPHAGENOME_API_KEY` and paste your key as the value. Toggle **Notebook access** on.
3. The code cell below reads the key via `google.colab.userdata.get('ALPHAGENOME_API_KEY')`.

In [ ]:
#@title Install & import dependencies { display-mode: "form" }
import os
from IPython.display import clear_output

if not os.path.exists('ALPHAGENOME_READY'):
    !pip install alphagenome
    open('ALPHAGENOME_READY', 'w').close()
    clear_output()
    print('Installation complete.')
else:
    print('Already installed, skipping.')

import dataclasses, json, re, requests, zipfile
from pathlib import Path
from typing import Any
from google.colab import userdata
from alphagenome.data import gene_annotation, genome, transcript as transcript_utils
from alphagenome.models import dna_client, variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

print('Ready.')

In [ ]:
#@title Setup: AlphaGenome initialization { display-mode: "form" }

def init(api_key, seq_length=dna_client.SEQUENCE_LENGTH_1MB):
    dna_model = dna_client.create(api_key)
    gtf = pd.read_feather('https://storage.googleapis.com/alphagenome/reference/gencode/hg38/gencode.v46.annotation.gtf.gz.feather')
    gtf_tx = gene_annotation.filter_protein_coding(gtf)
    gtf_tx = gene_annotation.filter_to_mane_select_transcript(gtf_tx)
    transcript_extractor = transcript_utils.TranscriptExtractor(gtf_tx)
    print(f'Model connected. {len(gtf)} GTF annotations loaded.')
    return dna_model, gtf, transcript_extractor

In [ ]:
#@title CREME experiment framework { display-mode: "form" }

_SEQ_CACHE = {}

def dinuc_shuffle(seq, num_shufs=None, rng=None):
    if rng is None: rng = np.random.RandomState()
    arr = np.frombuffer(bytearray(seq, 'utf8'), dtype=np.int8)
    chars, tokens = np.unique(arr, return_inverse=True)
    shuf_next_inds = []

    for t in range(len(chars)):
        mask = tokens[:-1] == t; inds = np.where(mask)[0]; shuf_next_inds.append(inds + 1)
    results = []

    for _ in range(num_shufs if num_shufs else 1):
        for t in range(len(chars)):
            inds = np.arange(len(shuf_next_inds[t]))
            if len(inds) > 1: inds[:-1] = rng.permutation(len(inds) - 1)
            shuf_next_inds[t] = shuf_next_inds[t][inds]
        counters = [0] * len(chars); ind = 0; result = np.empty_like(tokens); result[0] = tokens[ind]
        for j in range(1, len(tokens)):
            t = tokens[ind]; ind = shuf_next_inds[t][counters[t]]; counters[t] += 1; result[j] = tokens[ind]
        results.append(chars[result].tobytes().decode('ascii'))

    return results if num_shufs else results[0]

def _fetch_ref_seq(interval):
    key = f'{interval.chromosome}:{interval.start}-{interval.end}'
    if key in _SEQ_CACHE: 
        return _SEQ_CACHE[key]

    url = f'https://api.genome.ucsc.edu/getData/sequence?genome=hg38&chrom={interval.chromosome}&start={interval.start}&end={interval.end}'
    resp = requests.get(url, timeout=30); resp.raise_for_status()
    seq = resp.json()['dna'].upper(); _SEQ_CACHE[key] = seq; return seq

def _reverse_complement(seq):
    return seq.translate(str.maketrans('ACGT', 'TGCA'))[::-1]

def _make_shuffle_variant(tile, ref_seq, shuf_seq, name=None):
    return genome.Variant(chromosome=tile.chromosome, position=tile.start + 1, reference_bases=ref_seq, alternate_bases=shuf_seq, name=name or f'shuf_{tile.chromosome}:{tile.start}-{tile.end}')

def _average_shuffle_scores(dfs):
    combined = pd.concat(dfs, ignore_index=True)
    drop_cols = {'raw_score', 'quantile_score', 'variant_id', 'scored_interval'}

    for c in combined.columns:
        if c in drop_cols: continue
        try: combined[c].unique()
        except TypeError: drop_cols.add(c)
    group_cols = [c for c in combined.columns if c not in drop_cols]

    return combined.groupby(group_cols, as_index=False, sort=False, dropna=False).agg(raw_score=('raw_score', 'mean'), quantile_score=('quantile_score', 'mean'))

# --- Result dataclasses ---

@dataclasses.dataclass
class NecessityResult:
    tile_effects: pd.DataFrame; scores: pd.DataFrame; target_gene: str
    context_chrom: str; context_start: int; context_end: int; ontology_curie: str | None = None

    def to_csv(self, path, include_full_scores=False):
        self.tile_effects.to_csv(f'{path}_data.csv', index=False)
        meta = {'target_gene': self.target_gene, 'context_chrom': self.context_chrom, 'context_start': self.context_start, 'context_end': self.context_end, 'result_type': 'NecessityResult', 'ontology_curie': self.ontology_curie}
        with open(f'{path}_meta.json', 'w') as f: json.dump(meta, f, indent=2)
            
        if include_full_scores: self.scores.to_csv(f'{path}_full_scores.csv', index=False)

    @classmethod
    def from_csv(cls, path):
        tile_effects = pd.read_csv(f'{path}_data.csv')
        with open(f'{path}_meta.json') as f: meta = json.load(f)
        fp = Path(f'{path}_full_scores.csv'); scores = pd.read_csv(fp) if fp.exists() else pd.DataFrame()

        return cls(tile_effects=tile_effects, scores=scores, target_gene=meta['target_gene'], context_chrom=meta['context_chrom'], context_start=meta['context_start'], context_end=meta['context_end'], ontology_curie=meta.get('ontology_curie'))

@dataclasses.dataclass
class InteractionResult:
    rounds: pd.DataFrame; target_gene: str; ontology_curie: str

    def to_csv(self, path):
        self.rounds.to_csv(f'{path}_data.csv', index=False)
        meta = {'target_gene': self.target_gene, 'ontology_curie': self.ontology_curie, 'result_type': 'InteractionResult'}
        with open(f'{path}_meta.json', 'w') as f: json.dump(meta, f, indent=2)

    @classmethod
    def from_csv(cls, path):
        rounds = pd.read_csv(f'{path}_data.csv')
        with open(f'{path}_meta.json') as f: meta = json.load(f)

        return cls(rounds=rounds, target_gene=meta['target_gene'], ontology_curie=meta['ontology_curie'])

@dataclasses.dataclass
class CRISPRiResult:
    tile_scores: pd.DataFrame; tile_outputs: list; target_gene: str
    context_chrom: str; context_start: int; context_end: int

    def to_csv(self, path):
        self.tile_scores.to_csv(f'{path}_data.csv', index=False)
        meta = {'target_gene': self.target_gene, 'context_chrom': self.context_chrom, 'context_start': self.context_start, 'context_end': self.context_end, 'result_type': 'CRISPRiResult'}
        with open(f'{path}_meta.json', 'w') as f: json.dump(meta, f, indent=2)

    @classmethod
    def from_csv(cls, path):
        tile_scores = pd.read_csv(f'{path}_data.csv')
        with open(f'{path}_meta.json') as f: meta = json.load(f)

        return cls(tile_scores=tile_scores, tile_outputs=None, target_gene=meta['target_gene'], context_chrom=meta['context_chrom'], context_start=meta['context_start'], context_end=meta['context_end'])

# --- Cell type search ---

def search_cell_types(model, query, _cache={}):
    if 'all_biosamples' not in _cache:
        dummy_iv = genome.Interval('chr1', 0, dna_client.SEQUENCE_LENGTH_16KB)
        dummy_var = genome.Variant(chromosome='chr1', position=100, reference_bases='A', alternate_bases='T', name='dummy')
        result = model.score_variant(interval=dummy_iv, variant=dummy_var, variant_scorers=[variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']])
        df = variant_scorers.tidy_scores([result], match_gene_strand=True)
        bio_cols = [c for c in df.columns if c.startswith(('ontology_curie', 'biosample'))] or ['ontology_curie']
        _cache['all_biosamples'] = df[bio_cols].drop_duplicates().reset_index(drop=True)

    biosamples = _cache['all_biosamples']; q = query.lower()
    mask = biosamples.apply(lambda row: any(q in str(v).lower() for v in row), axis=1)

    return biosamples[mask].sort_values(biosamples.columns[0]).reset_index(drop=True)

# --- Main experiment class ---

class CREExperiment:
    def __init__(self, model, gene_symbol, gtf, transcript_extractor, ontology_terms, seq_length=dna_client.SEQUENCE_LENGTH_1MB):
        self.model, self.gene_symbol, self.gtf = model, gene_symbol, gtf
        self.transcript_extractor, self.ontology_terms, self.seq_length = transcript_extractor, ontology_terms, seq_length
        self.gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene_symbol)
        self.context_interval = self.gene_interval.resize(seq_length)

    @staticmethod
    def tile_region(region, tile_width=5000, step=None):
        if step is None: step = tile_width
        tiles = []; pos = region.start

        while pos < region.end:
            tiles.append(genome.Interval(region.chromosome, pos, min(pos + tile_width, region.end))); pos += step

        return tiles

    def necessity_test(self, tiles, scorers=None, target_gene=None, n_shuffles=10):
        target_gene = target_gene or self.gene_symbol
        if scorers is None: scorers = [variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']]
        rng = np.random.RandomState(42); per_tile_dfs = []

        for i, tile in enumerate(tqdm(tiles, desc='Necessity test')):
            ref_seq = _fetch_ref_seq(tile); shuffled_seqs = dinuc_shuffle(ref_seq, num_shufs=n_shuffles, rng=rng)
            replicate_dfs = []
            for s, shuf_seq in enumerate(shuffled_seqs):
                for orientation, seq in [('fwd', shuf_seq), ('rc', _reverse_complement(shuf_seq))]:
                    var = _make_shuffle_variant(tile, ref_seq, seq, name=f'Tile_{i+1}_shuf{s+1}_{orientation}')
                    res = self.model.score_variant(interval=self.context_interval, variant=var, variant_scorers=scorers)
                    replicate_dfs.append(variant_scorers.tidy_scores([res], match_gene_strand=True))
            avg_df = _average_shuffle_scores(replicate_dfs)
            avg_df['tile_idx'] = i; avg_df['tile_label'] = f'Tile {i+1}'
            avg_df['chrom'] = tile.chromosome; avg_df['tile_start'] = tile.start; avg_df['tile_end'] = tile.end
            per_tile_dfs.append(avg_df)

        scores_df = pd.concat(per_tile_dfs, ignore_index=True)
        gene_scores = scores_df[scores_df['gene_name'] == target_gene].copy()

        return NecessityResult(tile_effects=gene_scores, scores=scores_df, target_gene=target_gene, context_chrom=self.context_interval.chromosome, context_start=self.context_interval.start, context_end=self.context_interval.end)

    def interaction_test(self, tiles, scorers=None, target_gene=None, num_rounds=None, ontology_curie=None, n_shuffles=10):
        target_gene = target_gene or self.gene_symbol
        if scorers is None: scorers = [variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']]
        if num_rounds is None: num_rounds = len(tiles)
        num_rounds = min(num_rounds, len(tiles))
        remaining = list(range(len(tiles))); records = []; cumulative = 0.0

        for round_num in range(1, num_rounds + 1):
            nec = self.necessity_test([tiles[idx] for idx in remaining], scorers=scorers, target_gene=target_gene, n_shuffles=n_shuffles)
            effects = nec.tile_effects
            if ontology_curie is not None: 
                effects = effects[effects['ontology_curie'] == ontology_curie]
            elif len(effects) > 0: 
                ontology_curie = effects['ontology_curie'].iloc[0]
            if len(effects) == 0: 
                break
            worst_idx = effects['raw_score'].abs().idxmax(); worst_row = effects.loc[worst_idx]
            tile_num = int(worst_row['tile_label'].split()[-1]) - 1; original_idx = remaining[tile_num]; tile = tiles[original_idx]
            cumulative += worst_row['raw_score']
            records.append({'round': round_num, 'tile_removed': f'Tile {original_idx + 1}', 'tile_chrom': tile.chromosome, 'tile_start': tile.start, 'tile_end': tile.end, 'delta': float(worst_row['raw_score']), 'cumulative_effect': float(cumulative)})
            remaining.pop(tile_num)
            if not remaining: 
                break

        return InteractionResult(rounds=pd.DataFrame(records), target_gene=target_gene, ontology_curie=ontology_curie or '')

    def crispri_scan(self, tiles, outputs=None, target_gene=None, n_shuffles=10):
        target_gene = target_gene or self.gene_symbol
        if outputs is None: 
            outputs = [dna_client.OutputType.RNA_SEQ]
        rng = np.random.RandomState(42); tile_outputs = []; per_tile_dfs = []
        scorers = [variant_scorers.RECOMMENDED_VARIANT_SCORERS['RNA_SEQ']]

        for i, tile in enumerate(tqdm(tiles, desc='CRISPRi scan')):
            ref_seq = _fetch_ref_seq(tile); shuffled_seqs = dinuc_shuffle(ref_seq, num_shufs=n_shuffles, rng=rng)
            first_var = _make_shuffle_variant(tile, ref_seq, shuffled_seqs[0], name=f'CRISPRi_Tile_{i+1}_shuf1_fwd')
            voutput = self.model.predict_variant(interval=self.context_interval, variant=first_var, requested_outputs=outputs, ontology_terms=self.ontology_terms)
            tile_outputs.append((tile, first_var, voutput))
            replicate_dfs = []
            for s, shuf_seq in enumerate(shuffled_seqs):
                for orientation, seq in [('fwd', shuf_seq), ('rc', _reverse_complement(shuf_seq))]:
                    var = _make_shuffle_variant(tile, ref_seq, seq, name=f'CRISPRi_Tile_{i+1}_shuf{s+1}_{orientation}')
                    res = self.model.score_variant(interval=self.context_interval, variant=var, variant_scorers=scorers)
                    replicate_dfs.append(variant_scorers.tidy_scores([res], match_gene_strand=True))
            avg_df = _average_shuffle_scores(replicate_dfs)
            avg_df['tile_idx'] = i; avg_df['tile_label'] = f'Tile {i+1}'
            avg_df['chrom'] = tile.chromosome; avg_df['tile_start'] = tile.start; avg_df['tile_end'] = tile.end
            per_tile_dfs.append(avg_df)

        tile_scores = pd.concat(per_tile_dfs, ignore_index=True)
        
        return CRISPRiResult(tile_scores=tile_scores, tile_outputs=tile_outputs, target_gene=target_gene, context_chrom=self.context_interval.chromosome, context_start=self.context_interval.start, context_end=self.context_interval.end)

In [ ]:
#@title Plotting functions { display-mode: "form" }

def _format_pos(pos): return f'{pos:.3e}'

def _get_tile_scores(result, ontology_curie=None):
    df = result.tile_effects.copy()
    if ontology_curie is not None: 
        df = df[df['ontology_curie'] == ontology_curie]
    else: curie = df['ontology_curie'].iloc[0]; df = df[df['ontology_curie'] == curie]; ontology_curie = curie

    if len(df) > df['tile_idx'].nunique():
        keep = [c for c in ['tile_idx', 'tile_label', 'chrom', 'tile_start', 'tile_end'] if c in df.columns]
        df = df.groupby(keep, as_index=False, sort=False).agg(**{k: (k, 'mean') for k in ['raw_score', 'quantile_score'] if k in df.columns})
    return df.sort_values('tile_idx'), ontology_curie

def _bar_chart(df, chrom, title, ylabel='LFC (log2 ALT/REF)', score_col='raw_score', ax=None):
    if ax is None: _, ax = plt.subplots(figsize=(14, 4))

    for _, row in df.iterrows():
        ts, te = row['tile_start'], row['tile_end']; score = row[score_col]
        ax.bar((ts + te) / 2, score, width=(te - ts) * 0.9, color='#2166ac' if score < 0 else '#b2182b', edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8); ax.set_ylabel(ylabel); ax.set_xlabel(f'{chrom} position'); ax.set_title(title)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: _format_pos(int(x)))); ax.spines[['top', 'right']].set_visible(False); plt.tight_layout()

    return ax

def plot_necessity_bar(result, ontology_curie=None, ax=None):
    df, ontology_curie = _get_tile_scores(result, ontology_curie)

    return _bar_chart(df, df['chrom'].iloc[0], f'Necessity test: {result.target_gene} | {ontology_curie}', ax=ax)

def plot_necessity_heatmap(result, gtf=None, extra_curies=(), ax=None):
    df = result.tile_effects.copy().sort_values('tile_idx')
    primary_curie = getattr(result, 'ontology_curie', None)

    if extra_curies is not None:
        keep = set(extra_curies); (keep.add(primary_curie) if primary_curie else None)
        if keep: 
            df = df[df['ontology_curie'].isin(keep)]

    tiles_df = df[['tile_idx', 'chrom', 'tile_start', 'tile_end']].drop_duplicates().sort_values('tile_idx')
    num_tiles = len(tiles_df); tile_starts, tile_ends = tiles_df['tile_start'].values, tiles_df['tile_end'].values
    tile_widths = tile_ends - tile_starts; chrom = tiles_df['chrom'].iloc[0]
    agg_df = df.groupby(['ontology_curie', 'tile_idx'], as_index=False, sort=False).agg(raw_score=('raw_score', 'mean'))
    curies = sorted(agg_df['ontology_curie'].unique())
    if primary_curie and primary_curie in curies: 
        curies.remove(primary_curie); curies.insert(0, primary_curie)
    matrix = np.full((len(curies), num_tiles), np.nan)

    for _, row in agg_df.iterrows(): matrix[curies.index(row['ontology_curie']), int(row['tile_idx'])] = row['raw_score']
    region_start, region_end, tw = tile_starts.min(), tile_ends.max(), tile_widths[0]
    has_genes = gtf is not None

    if has_genes:
        fig, (ax_gene, ax_heat) = plt.subplots(2, 1, figsize=(14, max(3, 0.5 * len(curies) + 2)), height_ratios=[1, max(2, len(curies) * 0.6)], sharex=True)
    else:
        fig, ax_heat = plt.subplots(figsize=(14, max(3, 0.5 * len(curies) + 1)))
    vmax = np.nanmax(np.abs(matrix)) if not np.all(np.isnan(matrix)) else 1e-6

    for ci in range(len(curies)):
        for ti in range(num_tiles):
            val = matrix[ci, ti]
            if np.isnan(val): continue
            rect = mpatches.FancyBboxPatch((tile_starts[ti], ci - 0.4), tile_widths[ti], 0.8, boxstyle='round,pad=0', facecolor=plt.cm.RdBu_r((val / vmax + 1) / 2), edgecolor='white', linewidth=0.5)
            ax_heat.add_patch(rect)

    ax_heat.set_xlim(region_start - tw * 0.1, region_end + tw * 0.1); ax_heat.set_ylim(-0.5, len(curies) - 0.5)
    ax_heat.set_yticks(range(len(curies))); ax_heat.set_yticklabels(curies, fontsize=8)
    ax_heat.set_xlabel(f'{chrom} position'); ax_heat.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: _format_pos(int(x))))
    ax_heat.spines[['top', 'right']].set_visible(False)
    norm = mcolors.Normalize(vmin=-vmax, vmax=vmax); sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=norm); sm.set_array([])
    plt.colorbar(sm, ax=ax_heat, label='LFC', shrink=0.6, pad=0.02)

    if has_genes:
        pad = int((region_end - region_start) * 0.05); view_start, view_end = region_start - pad, region_end + pad
        genes_in_view = gtf[(gtf['Chromosome'] == chrom) & (gtf['End'] >= view_start) & (gtf['Start'] <= view_end)].copy()
        if 'Feature' in genes_in_view.columns: genes_in_view = genes_in_view[genes_in_view['Feature'] == 'gene']
        if 'gene_type' in genes_in_view.columns:
            pc = genes_in_view[genes_in_view['gene_type'] == 'protein_coding']
            if len(pc) > 0: genes_in_view = pc
        if 'gene_name' in genes_in_view.columns: 
            genes_in_view = genes_in_view.drop_duplicates(subset='gene_name')

        ax_gene.set_xlim(region_start - tw * 0.1, region_end + tw * 0.1); ax_gene.set_ylim(-0.5, max(1, len(genes_in_view)) - 0.5)
        ax_gene.set_yticks([]); ax_gene.set_title(f'Necessity: {result.target_gene} | {chrom}:{_format_pos(region_start)}-{_format_pos(region_end)}')
        ax_gene.spines[['top', 'right', 'bottom', 'left']].set_visible(False); ax_gene.tick_params(bottom=False)

        for yi, (_, gene) in enumerate(genes_in_view.iterrows()):
            g_start, g_end, strand = gene['Start'], gene['End'], gene.get('Strand', '+'); name = gene.get('gene_name', '')
            ax_gene.plot([g_start, g_end], [yi, yi], color='#333333', linewidth=2, solid_capstyle='butt')
            tss = g_start if strand == '+' else g_end; dx = (region_end - region_start) * 0.015 * (-1 if strand == '-' else 1)
            ax_gene.annotate('', xy=(tss + dx, yi), xytext=(tss, yi), arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))
            ax_gene.text((g_start + g_end) / 2, yi + 0.25, name, ha='center', va='bottom', fontsize=8, fontstyle='italic')

    plt.tight_layout(); return fig

def plot_necessity_genome(result, transcript_extractor=None, ontology_curie=None):
    df, ontology_curie = _get_tile_scores(result, ontology_curie)
    chrom = df['chrom'].iloc[0]; region_start, region_end = df['tile_start'].min(), df['tile_end'].max()
    pad = int((region_end - region_start) * 0.1); view_iv = genome.Interval(chrom, region_start - pad, region_end + pad)
    has_tx = transcript_extractor is not None

    if has_tx:
        fig, (ax_gene, ax_tiles) = plt.subplots(2, 1, figsize=(14, 4), height_ratios=[1, 2], sharex=True)
        transcripts = transcript_extractor.extract(view_iv); annot = plot_components.TranscriptAnnotation(transcripts)
        annot.plot_ax(ax_gene, axis_index=0, interval=view_iv); ax_gene.set_title(f'Necessity: {result.target_gene} tiles')
        ax_gene.spines[['top', 'right', 'bottom']].set_visible(False); ax_gene.tick_params(bottom=False)
    else: 
        fig, ax_tiles = plt.subplots(figsize=(14, 2))

    scores, tile_starts, tile_ends = df['raw_score'].values, df['tile_start'].values, df['tile_end'].values
    vmax = max(abs(scores.min()), abs(scores.max())) if len(scores) else 1e-6
    norm = mcolors.Normalize(vmin=-vmax, vmax=vmax); cmap = plt.cm.RdBu_r

    for i in range(len(df)):
        ts, te, tw = tile_starts[i], tile_ends[i], tile_ends[i] - tile_starts[i]
        rect = mpatches.FancyBboxPatch((ts, 0.1), tw, 0.8, boxstyle='round,pad=0', facecolor=cmap(norm(scores[i])), edgecolor='black', linewidth=0.5)
        ax_tiles.add_patch(rect); ax_tiles.text(ts + tw / 2, 0.5, df['tile_label'].values[i], ha='center', va='center', fontsize=7)
    ax_tiles.set_xlim(view_iv.start, view_iv.end); ax_tiles.set_ylim(0, 1); ax_tiles.set_xlabel(f'{chrom} position'); ax_tiles.set_yticks([])
    ax_tiles.set_title(f'Tile necessity scores ({ontology_curie})')
    ax_tiles.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: _format_pos(int(x)))); ax_tiles.spines[['top', 'right', 'left']].set_visible(False)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([]); plt.colorbar(sm, ax=ax_tiles, label='LFC', shrink=0.6, pad=0.02)
    plt.tight_layout(); return fig

def plot_interaction_waterfall(result, ax=None):
    if ax is None: _, ax = plt.subplots(figsize=(8, 5))
    rdf = result.rounds if isinstance(result.rounds, pd.DataFrame) else pd.DataFrame(result.rounds)
    rounds_num = [0] + rdf['round'].tolist(); cum = [0.0] + rdf['cumulative_effect'].tolist()
    ax.step(rounds_num, cum, where='post', color='#2166ac', linewidth=2, marker='o', markersize=6)
    ax.fill_between(rounds_num, cum, step='post', alpha=0.15, color='#2166ac')

    for _, r in rdf.iterrows():
        ax.annotate(r['tile_removed'], xy=(r['round'], r['cumulative_effect']), xytext=(5, 10), textcoords='offset points', fontsize=8, ha='left', arrowprops=dict(arrowstyle='->', color='grey', lw=0.8))

    ax.set_xlabel('Tiles removed'); ax.set_ylabel('Cumulative LFC from WT')
    ax.set_title(f'Interaction test: {result.target_gene} | {result.ontology_curie}')
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--'); ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); return ax

def plot_interaction_contribution(result, ax=None):
    rdf = result.rounds if isinstance(result.rounds, pd.DataFrame) else pd.DataFrame(result.rounds)

    return _bar_chart(rdf, rdf['tile_chrom'].iloc[0], f'Per-round contribution: {result.target_gene} | {result.ontology_curie}', ylabel='Delta LFC', score_col='delta', ax=ax)

def plot_crispri_summary(result, ontology_curie=None, ax=None):
    df = result.tile_scores.copy(); df = df[df['gene_name'] == result.target_gene]
    if ontology_curie is not None: 
        df = df[df['ontology_curie'] == ontology_curie]
    elif 'ontology_curie' in df.columns and len(df) > 0: 
        curie = df['ontology_curie'].iloc[0]; df = df[df['ontology_curie'] == curie]; ontology_curie = curie
    keep = [c for c in ['tile_idx', 'tile_label', 'chrom', 'tile_start', 'tile_end'] if c in df.columns]
    df = df.groupby(keep, as_index=False, sort=False).agg(raw_score=('raw_score', 'mean')).sort_values('tile_idx')

    return _bar_chart(df, df['chrom'].iloc[0], f'CRISPRi scan: {result.target_gene}' + (f' | {ontology_curie}' if ontology_curie else ''), ax=ax)

def plot_crispri_tracks(result, tile_idx, modality='rna_seq', transcript_extractor=None, zoom_width=2**15):
    if result.tile_outputs is None: 
        raise ValueError('tile_outputs is None (not available from CSV). Track plotting requires original tile_outputs.')
    tile, variant, voutput = result.tile_outputs[tile_idx]
    ctx_iv = genome.Interval(result.context_chrom, result.context_start, result.context_end)
    ref_tdata, alt_tdata = getattr(voutput.reference, modality), getattr(voutput.alternate, modality)
    components = []

    if transcript_extractor is not None:
        components.append(plot_components.TranscriptAnnotation(transcript_extractor.extract(ctx_iv)))
    components.append(plot_components.OverlaidTracks(tdata={'WT': ref_tdata, 'CRISPRi': alt_tdata}, colors={'WT': 'dimgrey', 'CRISPRi': 'red'}))
    view_iv = ref_tdata.interval.resize(zoom_width) if zoom_width else ref_tdata.interval
    plot_components.plot(components, interval=view_iv, annotations=[plot_components.VariantAnnotation([variant])], title=f'CRISPRi Tile {tile_idx+1}: {tile} | {modality}')
    plt.show()

In [ ]:
#@title Connect to AlphaGenome { display-mode: "form" }
dna_model, gtf, transcript_extractor = init(api_key=userdata.get('ALPHAGENOME_API_KEY'))

In [ ]:
#@title Search cell types (optional helper) { display-mode: "form" }
search_query = 'CD34'  #@param {type:"string"}
search_cell_types(dna_model, search_query)

In [ ]:
#@title 1. Input gene and cell type { display-mode: "form" }
gene = 'TAL1'  #@param {type:"string"}
#@markdown > Use the **Search cell types** helper above to find your ontology CURIE, then paste it here:
ontology_curie = 'CL:0001059'  #@param {type:"string"}

gene = gene.strip()
ontology_curie = ontology_curie.strip()

myexp = CREExperiment(
    model=dna_model, gene_symbol=gene, gtf=gtf,
    transcript_extractor=transcript_extractor, ontology_terms=[ontology_curie],
)

gene_iv = gene_annotation.get_gene_interval(gtf, gene_symbol=gene)
print(f'Gene: {gene}')
print(f'Gene interval: {gene_iv}')
print(f'Ontology: {ontology_curie}')

In [ ]:
#@title 2. Advanced settings { display-mode: "form" }

#@markdown ### Tiling parameters
#@markdown **Tile width** — size of each perturbation window in base pairs. Smaller tiles give finer resolution but cost more API calls. 5 kb is a good default for identifying enhancer-scale elements.
tile_width_bp = 5000  #@param [1000, 2000, 5000, 10000] {type:"raw"}
#@markdown **Shuffle replicates** — how many independent dinucleotide shuffles to average per tile. More replicates reduce noise but increase runtime. Each replicate is scored in both forward and reverse-complement orientations (so 2 = 4 API calls per tile).
shuffle_replicates = 2  #@param [1, 2, 5, 10] {type:"raw"}
#@markdown **Scan region width** — total genomic window (in bp) centered on the gene to tile. 40 kb covers most proximal enhancers; increase to 100–200 kb to capture distal elements.
scan_region_width_bp = 40000  #@param {type:"integer"}

#@markdown ### Experiments to run
#@markdown Uncheck any experiment to skip it during "Run All".
run_necessity = True  #@param {type:"boolean"}
run_interaction = True  #@param {type:"boolean"}
run_crispri = True  #@param {type:"boolean"}

#@markdown ### Interaction test settings
#@markdown **Ablation rounds** — how many tiles to remove in the greedy interaction test. Each round removes the most impactful remaining tile and re-scores. More rounds reveal deeper cooperativity but take longer.
greedy_ablation_rounds = 4  #@param [2, 3, 4, 5, 6] {type:"raw"}

#@markdown ### CRISPRi scan settings
#@markdown **Target region** — the genomic region to scan at fine resolution. Leave blank to auto-detect (picks the region around the most necessary tile from the necessity test ± 4 kb). Or enter a specific region.
crispri_target_region = ''  #@param {type:"string"}
#@markdown **CRISPRi tile width** — size of each fine-resolution tile in bp. 1 kb mimics a typical CRISPRi guide tiling screen.
crispri_tile_width_bp = 1000  #@param [500, 1000, 2000] {type:"raw"}
#@markdown **Output tracks** — which genomic assay tracks to predict for each perturbation. RNA-seq is always included; add DNASE or ATAC for chromatin accessibility.
crispri_output_tracks = "RNA_SEQ"  #@param ["RNA_SEQ", "RNA_SEQ + DNASE", "RNA_SEQ + ATAC"]

# Compute tiles
tile_region = gene_iv.resize(scan_region_width_bp)
tiles = CREExperiment.tile_region(tile_region, tile_width=tile_width_bp)

print(f'Tile region: {tile_region} ({tile_region.width} bp)')
print(f'Number of tiles: {len(tiles)} (each {tile_width_bp} bp)')
for i, t in enumerate(tiles):
    print(f'  Tile {i+1}: {t}')

## Necessity Test

**Concept** ([CREME](https://www.nature.com/articles/s41588-024-01923-3), Toneyan and Koo, Nature Genetics 2024): Tile a locus into fixed-width windows, replace each tile with a dinucleotide-preserving shuffle, and measure the effect on target gene expression. Tiles whose perturbation strongly reduces expression are **necessary** CREs.

In [ ]:
#@title 3. Run necessity test { display-mode: "form" }
if run_necessity:
    necessity = myexp.necessity_test(tiles=tiles, target_gene=gene, n_shuffles=shuffle_replicates)
    print(f'Total score rows: {len(necessity.scores)}')
    print(f'Tile effects for {gene}: {len(necessity.tile_effects)} rows')
    necessity.tile_effects[['tile_label', 'ontology_curie', 'raw_score', 'quantile_score']].head(20)
else:
    print('Necessity test skipped.')

In [ ]:
#@title Necessity — bar chart { display-mode: "form" }
if run_necessity:
    plot_necessity_bar(necessity, ontology_curie=ontology_curie)
    plt.show()

In [ ]:
#@title Necessity — heatmap { display-mode: "form" }
if run_necessity:
    plot_necessity_heatmap(necessity, gtf=gtf)
    plt.show()

In [ ]:
#@title Necessity — genome browser view { display-mode: "form" }
if run_necessity:
    plot_necessity_genome(necessity, transcript_extractor=transcript_extractor, ontology_curie=ontology_curie)
    plt.show()

## Higher-Order Interaction Test

**Concept**: Starting from the necessity results, run **greedy ablation** (CREME's `higher_order_interaction_test`). At each round, the tile with the largest remaining effect is permanently removed, and all other tiles are re-scored. This reveals how CRE contributions compound — e.g., whether removing the MuTE enhancer exposes dependency on a secondary element.

Under a linear-additivity approximation, each tile is scored independently per round and the cumulative LFC is accumulated. This is the same greedy strategy CREME uses, adapted for AlphaGenome's variant API.

In [ ]:
#@title 4. Run higher-order interaction test { display-mode: "form" }
if run_interaction:
    interaction = myexp.interaction_test(
        tiles=tiles, target_gene=gene, num_rounds=greedy_ablation_rounds, n_shuffles=shuffle_replicates,
    )
else:
    print('Interaction test skipped.')

In [ ]:
#@title Interaction — waterfall plot { display-mode: "form" }
if run_interaction:
    plot_interaction_waterfall(interaction)
    plt.show()

In [ ]:
#@title Interaction — contribution plot { display-mode: "form" }
if run_interaction:
    plot_interaction_contribution(interaction)
    plt.show()

## CRISPRi Tiling Scan

**Concept**: A focused, fine-resolution perturbation scan of a specific regulatory region. This mirrors CRISPRi tiling screens — tile a candidate enhancer with small windows, replace each with a dinucleotide shuffle, and compare full WT vs perturbed tracks.

For each perturbation we get:
- **Scalar scores** (GeneMaskLFCScorer) for quantitative comparison (averaged over shuffle replicates)
- **Full REF/ALT tracks** (predict_variant) for visual inspection of RNA-seq changes

In [ ]:
#@title 5. Run CRISPRi tiling scan { display-mode: "form" }
if run_crispri:
    crispri_region_str = crispri_target_region.strip()
    
    if crispri_region_str:
        m = re.match(r'^(chr[\w]+):(\d+)-(\d+)$', crispri_region_str)
        if not m: raise ValueError(f'Invalid region format: {crispri_region_str!r}. Expected chr1:12345-67890')
        scan_region = genome.Interval(m.group(1), int(m.group(2)), int(m.group(3)))
    elif run_necessity:
        target_effects = necessity.tile_effects[necessity.tile_effects['ontology_curie'] == ontology_curie]
        worst_idx = target_effects['raw_score'].idxmin()
        worst_label = target_effects.loc[worst_idx, 'tile_label']; ti = int(worst_label.split()[-1]) - 1; worst_tile = tiles[ti]
        scan_region = genome.Interval(worst_tile.chromosome, worst_tile.start - 4000, worst_tile.end + 4000)
        print(f'Auto-detected scan region around {worst_label}: {scan_region}')
    else:
        raise ValueError('No CRISPRi region specified and necessity test was skipped. Please enter a region in Advanced settings.')

    crispri_tiles = CREExperiment.tile_region(scan_region, tile_width=crispri_tile_width_bp)
    print(f'CRISPRi scan region: {scan_region} ({scan_region.width} bp)')
    print(f'Number of {crispri_tile_width_bp}bp tiles: {len(crispri_tiles)}')

    output_map = {'RNA_SEQ': [dna_client.OutputType.RNA_SEQ], 'RNA_SEQ + DNASE': [dna_client.OutputType.RNA_SEQ, dna_client.OutputType.DNASE], 'RNA_SEQ + ATAC': [dna_client.OutputType.RNA_SEQ, dna_client.OutputType.ATAC]}
    crispri = myexp.crispri_scan(tiles=crispri_tiles, outputs=output_map[crispri_output_tracks], target_gene=gene, n_shuffles=shuffle_replicates)
    print(f'CRISPRi scan complete. {len(crispri.tile_outputs)} tiles scored.')
else:
    print('CRISPRi scan skipped.')

In [ ]:
#@title CRISPRi — summary bar chart { display-mode: "form" }
if run_crispri:
    plot_crispri_summary(crispri)
    plt.show()

In [ ]:
#@title CRISPRi — track view (auto-selects best tile) { display-mode: "form" }
if run_crispri:
    gene_scores = crispri.tile_scores[crispri.tile_scores['gene_name'] == gene]
    if len(gene_scores) > 0:
        best_idx = gene_scores['raw_score'].abs().idxmax(); best_label = gene_scores.loc[best_idx, 'tile_label']
        ti = int(best_label.split()[-1]) - 1
        print(f'Most impactful tile: {best_label} ({crispri_tiles[ti]})')
        print(f'LFC = {gene_scores.loc[best_idx, "raw_score"]:.4f}')
        plot_crispri_tracks(crispri, tile_idx=ti, modality='rna_seq', transcript_extractor=transcript_extractor, zoom_width=2**15)
    else:
        print(f'{gene} not found in CRISPRi scores. Check context interval.')

In [ ]:
#@title 6. Download results { display-mode: "form" }
from google.colab import files

result_dir = f'{gene}_{ontology_curie.replace(":", "_")}_results'
os.makedirs(result_dir, exist_ok=True)
saved = []
if run_necessity: necessity.to_csv(f'{result_dir}/necessity'); saved.append('necessity')
if run_interaction: interaction.to_csv(f'{result_dir}/interaction'); saved.append('interaction')
if run_crispri: crispri.to_csv(f'{result_dir}/crispri'); saved.append('crispri')

zip_name = f'{gene}_{ontology_curie.replace(":", "_")}.results.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(result_dir):
        for fn in filenames: zf.write(os.path.join(root, fn), os.path.relpath(os.path.join(root, fn), '.'))

print(f'Saved experiments: {", ".join(saved)}')
print(f'Downloading {zip_name}...')
files.download(zip_name)

---
## Appendix: Quick Reference

| Task | API Method | Key Parameters |
|------|-----------|----------------|
| Predict from sequence | `dna_model.predict_sequence()` | `sequence`, `requested_outputs`, `ontology_terms` |
| Predict from interval | `dna_model.predict_interval()` | `interval`, `requested_outputs`, `ontology_terms` |
| Predict variant (REF vs ALT) | `dna_model.predict_variant()` | `interval`, `variant`, `requested_outputs`, `ontology_terms` |
| Score a variant | `dna_model.score_variant()` | `interval`, `variant`, `variant_scorers` |
| Score many variants | `dna_model.score_variants()` | `intervals`, `variants`, `variant_scorers` |
| ISM | `dna_model.score_ism_variants()` | `interval`, `ism_interval`, `variant_scorers` |
| **CREME necessity** | `CREExperiment.necessity_test()` | `tiles`, `target_gene`, `n_shuffles` |
| **CREME interaction** | `CREExperiment.interaction_test()` | `tiles`, `num_rounds`, `ontology_curie`, `n_shuffles` |
| **CREME CRISPRi scan** | `CREExperiment.crispri_scan()` | `tiles`, `outputs`, `target_gene`, `n_shuffles` |
| **Cell type search** | `search_cell_types()` | `model`, `query` |

### Supported sequence lengths
- `SEQUENCE_LENGTH_16KB` = 16,384 bp
- `SEQUENCE_LENGTH_100KB` = 131,072 bp
- `SEQUENCE_LENGTH_500KB` = 524,288 bp
- `SEQUENCE_LENGTH_1MB` = 1,048,576 bp

### Key variant scorers
- `GeneMaskLFCScorer` — Log fold change per gene (gene-centric, signed)
- `CenterMaskScorer` — Effect in a window around the variant (non-gene-centric)
- `GeneMaskSplicingScorer` — Splicing changes per gene
- `SpliceJunctionScorer` — Splice junction disruption
- `ContactMapScorer` — 3D chromatin contact disruption
- `PolyadenylationScorer` — paQTL effects

### Ontology resources
- UBERON (anatomy): https://www.ebi.ac.uk/ols4/ontologies/uberon
- Cell Ontology: https://www.ebi.ac.uk/ols4/ontologies/cl
- EFO (cell lines): https://www.ebi.ac.uk/ols4/ontologies/efo

### Documentation
- Full docs: https://www.alphagenomedocs.com/
- Variant scoring: https://www.alphagenomedocs.com/variant_scoring.html
- Visualization: https://www.alphagenomedocs.com/visualization_library_basics.html